# USGS LandsatLook — Landsat C2 requester-pays

`usgs-landsat` is the **authoritative** USGS STAC for Landsat Collection 2. Earth Search
mirrors the same data anonymously, but USGS keeps SR / ST / L1 as separate collections and
serves them from a **requester-pays** bucket (`s3://usgs-landsat`) — so reads need an AWS
identity and the bucket owner stays uncharged.

Earthlens wires this in via a per-collection `signer: aws-requester-pays` override on every
USGS LandsatLook row (the same pattern Earth Search uses for its `landsat-c2-l2` mirror). The
notebook below is a **recipe**: it constructs the request, but only runs `download()` when
`AWS_ACCESS_KEY_ID` + `AWS_SECRET_ACCESS_KEY` are set in the environment.

## The endpoint + curated collections (no network)

In [ ]:
from earthlens.stac import Catalog

cat = Catalog()
e = cat.endpoints["usgs-landsat"]
print(f"url   : {e.url}")
print(f"signer: {e.signer}")
print(f"region: {e.region}")
for key in (
    "usgs-landsat/landsat-c2l2-sr",
    "usgs-landsat/landsat-c2l2-st",
    "usgs-landsat/landsat-c2l1",
):
    col = cat.get_collection(key)
    print(f"  {key}  signer={col.signer}  default={col.default_assets}")

## Surface Reflectance over SF Bay (gated on AWS creds)

Same request shape as Earth Search — only `data_source` and `dataset` change. If your
environment has no AWS keys, the recipe prints the request and stops; with keys it pulls a
single Landsat-9 surface-reflectance scene as a 4-band COG.

In [ ]:
import os
import tempfile

from earthlens.core import EarthLens

request = dict(
    data_source="usgs-landsat",  # alias 'landsat' works too
    start="2024-06-01",
    end="2024-08-31",
    dataset="usgs-landsat/landsat-c2l2-sr",
    variables=["red", "green", "blue", "nir08"],
    aoi=[-122.5, 37.5, -122.0, 38.0],  # SF Bay
    path=tempfile.mkdtemp(),
    max_items=1,
)
have_aws = bool(os.environ.get("AWS_ACCESS_KEY_ID")) and bool(
    os.environ.get("AWS_SECRET_ACCESS_KEY")
)
print("AWS creds present:", have_aws)
if have_aws:
    paths = EarthLens(**request).download()
    print("wrote:", paths)
else:
    paths = []
    print("set AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY to run; recipe shown above.")

## Inspect (only when the read ran with AWS creds)

In [ ]:
if paths:
    from pyramids.dataset import Dataset

    ds = Dataset.read_file(paths[0])
    print(f"bands={ds.band_count} epsg={ds.epsg} shape={ds.read_array().shape}")

## Variants

- **Surface Temperature**: switch `dataset` to `usgs-landsat/landsat-c2l2-st` and `variables`
  to `["lwir"]` (LST band).
- **Level-1 TOA**: `dataset="usgs-landsat/landsat-c2l1"` exposes `pan`, `cirrus`, and
  `lwir11`/`lwir12` alongside the spectral bands.
- **Earth Search mirror**: `data_source="earth-search"`,
  `dataset="earth-search/landsat-c2-l2"` reads the same data — also requester-pays, since the
  Earth Search row carries the same per-collection signer override.